# Practical 8 — Decision Trees and Ensemble Methods

**Lectures:** 16. Tree-based Learning · 17. Ensemble Methods
(with grid search from 20. Cross Validation and Hyperparameter Tuning)

1. Implement **entropy**, **Gini impurity** and **information gain**, and the search for the best split.
2. Build a small **decision-tree classifier from scratch** (recursive, greedy, depth-limited) and compare it with scikit-learn.
3. See how tree **depth** controls over- and underfitting, and try a **regression tree**.
4. Implement **bagging** yourself, then compare bagging, **random forest**, **AdaBoost**, **gradient boosting** and **stacking**.
5. Tune a random forest with `GridSearchCV`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

## 1. Impurity measures and information gain (≈20 min)
For a node whose examples have class proportions $p_1,\dots,p_K$:
$\;H = -\sum_k p_k\log_2 p_k$ (entropy), $\;G = 1-\sum_k p_k^2$ (Gini).
Splitting a node $S$ into children $S_L, S_R$ gives
$\text{IG} = I(S) - \frac{|S_L|}{|S|}I(S_L) - \frac{|S_R|}{|S|}I(S_R)$ for impurity $I$.

**TODO 1.1** Implement the three functions. Check: `entropy([0,0,1,1]) == 1`, `gini([0,0,1,1]) == 0.5`,
and both are 0 for a pure node. Plot entropy and Gini of a binary node against $p$.

In [ ]:
def entropy(y):
    # YOUR CODE HERE
    raise NotImplementedError

def gini(y):
    # YOUR CODE HERE
    raise NotImplementedError

def information_gain(y, y_left, y_right, impurity=entropy):
    # YOUR CODE HERE
    raise NotImplementedError

print(entropy([0, 0, 1, 1]), gini([0, 0, 1, 1]), entropy([1, 1, 1]), gini([1, 1, 1]))
# YOUR CODE HERE

**TODO 1.2** Worked example. The toy table below records whether students passed a quiz.
Compute the information gain (entropy) of splitting on each of the three binary features.
Which feature would the tree put at the root?

In [ ]:
toy = pd.DataFrame({
    "attended_lectures": [1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0],
    "did_practicals":    [1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1],
    "morning_person":    [0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0],
    "passed":            [1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0],
})
# YOUR CODE HERE

## 2. A decision tree from scratch (≈35 min)
For numerical features, candidate thresholds are the midpoints between consecutive sorted distinct values.

**TODO 2.1** Write `best_split(X, y)` returning `(feature, threshold, gain)` for the split that maximises
information gain (Gini), or `None` if no split improves impurity.

**TODO 2.2** Complete the recursive `build_tree` and `predict_one`. A node is a `dict`: a leaf is
`{"leaf": True, "value": majority_class}`; an internal node is
`{"leaf": False, "feature": j, "threshold": t, "left": ..., "right": ...}` with `x[j] <= t` going left.
Stop when the node is pure, when `depth == max_depth`, or when it has fewer than `min_samples_split` examples.

In [ ]:
def best_split(X, y, impurity=gini):
    best = None
    # YOUR CODE HERE
    return best

def majority(y):
    return Counter(y.tolist()).most_common(1)[0][0]

def build_tree(X, y, depth=0, max_depth=3, min_samples_split=2):
    # YOUR CODE HERE
    raise NotImplementedError

def predict_one(node, x):
    # YOUR CODE HERE
    raise NotImplementedError

def predict_tree(node, X):
    return np.array([predict_one(node, x) for x in X])

def print_tree(node, names, indent="", prefix=""):
    if node["leaf"]:
        print(indent + prefix + f"class {node['value']}"); return
    print(indent + prefix + f"{names[node['feature']]} <= {node['threshold']:.3f}?")
    print_tree(node["left"], names, indent + "    ", "yes → ")
    print_tree(node["right"], names, indent + "    ", "no  → ")

**TODO 2.3** Train your tree (max depth 3) on the banknote data and print it. Compare its test accuracy and its
root split with `DecisionTreeClassifier(max_depth=3, random_state=0)` (visualise the latter with `plot_tree`).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

bank = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/bill_authentication.csv")
Xb, yb = bank.drop(columns="Class").to_numpy(), bank["Class"].to_numpy()
names = list(bank.columns[:-1])
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)

# YOUR CODE HERE

## 3. Depth, overfitting and regression trees (≈15 min)
**TODO 3.1** Using scikit-learn trees, plot training and test accuracy against `max_depth` = 1…15 on the
**white wine** data, turned into a binary problem: *good* wine = quality ≥ 7 (about 22 % of wines).
Where does the tree start to overfit?

In [ ]:
wine = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/winequality-white.csv", sep=";")
Xw = wine.drop(columns="quality").to_numpy(); yw = (wine["quality"] >= 7).astype(int).to_numpy()
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.25, stratify=yw, random_state=0)
print("positive rate:", yw.mean().round(3))
# YOUR CODE HERE

**TODO 3.2** A regression tree predicts the **mean** target of the training examples in each leaf and chooses splits
that minimise the squared error. Fit `DecisionTreeRegressor` with `max_depth` 2 and 5 to the petrol-consumption
data below using the single feature `Average_income` and plot the (step-shaped) predictions. Compare the
test RMSE of a depth-3 tree using all features with `LinearRegression`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

petrol = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/petrol_consumption.csv")
print(petrol.head())
# YOUR CODE HERE

## 4. Ensembles (≈30 min)
**TODO 4.1 — Bagging from scratch.** Train $B=50$ fully-grown `DecisionTreeClassifier`s, each on a **bootstrap
sample** (draw $m$ examples *with replacement*) of the wine training set, and predict by **majority vote**.
Plot test accuracy against the number of trees used in the vote (1…50) and compare with a single tree.

In [ ]:
# YOUR CODE HERE

**TODO 4.2 — Compare ensemble methods.** Using 5-fold stratified cross-validation on the wine **training** set, compare
(with default settings unless stated): a single `DecisionTreeClassifier`, `BaggingClassifier(n_estimators=100)`,
`RandomForestClassifier(n_estimators=200)`, `AdaBoostClassifier(n_estimators=200)`,
`GradientBoostingClassifier()`, and a `StackingClassifier` whose base learners are a random forest, k-NN (with scaling) and
logistic regression (with scaling), with logistic regression as the final estimator. Report mean ± s.d. of
**accuracy** and **F1** for each. Set `random_state=0` wherever available. Which is best? Why can random forests
beat plain bagging?

In [ ]:
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, StackingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=0)
# YOUR CODE HERE

## 5. Hyperparameter tuning with grid search (≈15 min)
**TODO 5.1** Use `GridSearchCV` (5-fold stratified, `scoring="f1"`) to tune a random forest over
`n_estimators ∈ {100, 300}`, `max_features ∈ {"sqrt", 0.5}`, `min_samples_leaf ∈ {1, 3}`. Report the best parameters and
CV score, then — **once** — the test accuracy, F1 and confusion matrix of the refitted best model. Show the five most
important features (`feature_importances_`).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, confusion_matrix

# YOUR CODE HERE

## 6. Extension
* Add `max_depth` to your from-scratch tree's competitors: implement **pruning by validation** or a `min_gain` threshold.
* Implement **AdaBoost** yourself with decision stumps (`DecisionTreeClassifier(max_depth=1)` and `sample_weight`), following the lecture's weight-update rule.

## Lab report checklist (Practical 8)
1. Your information-gain calculations for the toy table and the chosen root feature.
2. Your from-scratch tree (printed) and its test accuracy versus scikit-learn's.
3. The accuracy-vs-depth figure and where overfitting begins; the regression-tree figure.
4. The bagging curve, and the table comparing six models (accuracy and F1, mean ± s.d.). Explain *why* ensembles help, distinguishing bagging from boosting from stacking.
5. Grid-search result and final test performance. Why is F1 a more sensible selection criterion than accuracy here?